## Phase 4 - Model Training

Training baseline models on the TF-IDF features and comparing with the validation split.

In [1]:
import pickle
from pathlib import Path

import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

## Loading Vectorizer and Preparing Feature Matrices

In [2]:
train_df = pd.read_csv("../data/processed/train_data.csv")
val_df = pd.read_csv("../data/processed/validation_data.csv")
test_df = pd.read_csv("../data/processed/test_data.csv")

with open("../models/tfidf_vectorizer.pkl", "rb") as file_handle:
    vectorizer = pickle.load(file_handle)

for frame in (train_df, val_df, test_df):
    frame["cleaned_text"] = frame["cleaned_text"].fillna("").astype(str)

X_train = vectorizer.transform(train_df["cleaned_text"])
X_val = vectorizer.transform(val_df["cleaned_text"])
X_test = vectorizer.transform(test_df["cleaned_text"])
y_train = train_df["class"]
y_val = val_df["class"]
y_test = test_df["class"]

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)
print("Training labels:")
print(y_train.value_counts().to_string())

Train shape: (185658, 20000)
Validation shape: (23208, 20000)
Test shape: (23208, 20000)
Training labels:
class
suicide        92829
non-suicide    92829


## Training and Evaluating All Classification Models

In [3]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "naive_bayes": MultinomialNB(),
    "linear_svm": LinearSVC(class_weight="balanced", random_state=42),
}

calibrated_svm = CalibratedClassifierCV(LinearSVC(class_weight="balanced", random_state=42), method="sigmoid", cv=3)
models["calibrated_svm"] = calibrated_svm

voting_model = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        ("nb", MultinomialNB()),
        ("svm", LinearSVC(class_weight="balanced", random_state=42)),
    ],
    voting="hard",
)
models["voting_classifier"] = voting_model

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

results = []
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_val, predictions),
        "precision": precision_score(y_val, predictions, pos_label="suicide"),
        "recall": recall_score(y_val, predictions, pos_label="suicide"),
        "f1": f1_score(y_val, predictions, pos_label="suicide"),
    }
    results.append(metrics)
    trained_models[name] = model

    model_path = models_dir / f"{name}.pkl"
    with open(model_path, "wb") as file_handle:
        pickle.dump(model, file_handle)
    print(f"Saved: {model_path}")
    print(classification_report(y_val, predictions))

results_df = pd.DataFrame(results).sort_values(by="f1", ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

Training logistic_regression...
Saved: ../models/logistic_regression.pkl
              precision    recall  f1-score   support

 non-suicide       0.93      0.95      0.94     11604
     suicide       0.94      0.93      0.94     11604

    accuracy                           0.94     23208
   macro avg       0.94      0.94      0.94     23208
weighted avg       0.94      0.94      0.94     23208

Training naive_bayes...
Saved: ../models/naive_bayes.pkl
              precision    recall  f1-score   support

 non-suicide       0.95      0.87      0.91     11604
     suicide       0.88      0.95      0.91     11604

    accuracy                           0.91     23208
   macro avg       0.91      0.91      0.91     23208
weighted avg       0.91      0.91      0.91     23208

Training linear_svm...
Saved: ../models/linear_svm.pkl
              precision    recall  f1-score   support

 non-suicide       0.93      0.95      0.94     11604
     suicide       0.95      0.93      0.94     1160

## Best Model Selection

In [4]:
best_model_name = results_df.loc[0, "model"]
best_model = trained_models[best_model_name]
print("Best validation model:", best_model_name)
print("Validation F1:", results_df.loc[0, "f1"])


Best validation model: voting_classifier
Validation F1: 0.9398350105817821
